# Custom Climate Profiles Generation

#### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
import pandas as pd
import xarray as xr
import numpy as np

from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import get_subsetting_options

from climakitae.explore.extreme_meteorological_year import (
    persistence_XMY,
    shock_XMY,
    persistence_get_top_hours)

from climakitae.core.constants import UNSET

from climakitae.explore.typical_meteorological_year import (
    get_cdf,
    get_cdf_monthly,
)


import warnings

warnings.filterwarnings("ignore")

### Generate an Extreme Meteorological Year (XMY) Profile

In [ ]:
start_year = 2005
end_year = 2020

In [ ]:
# Set up the TMY profile generator! The verbose option will output progress of the TMY generation
xmy = persistence_XMY(
    ## Location -- uncomment your desired option
    station_name = "Los Angeles International Airport (KLAX)",
    # latitude = latitude,
    # longitude = longitude,
    q = 0.9,
    
    # Approach -- uncomment your desired option
    warming_level = 1.5,
    # start_year = start_year,
    # end_year = end_year,
    verbose=True,
)

# Generate the profile!
xmy.generate_xmy()

**Export to non-EPW format.** TMY profiles are exported in .epw format by default, but can be exported as both `.csv` and `.tmy` file formats using the method `export_tmy_data` with the argument `extension="csv"` as shown below.

In [ ]:
xmy.export_xmy_data(extension="csv")

### Unit Testing

In [4]:
air_temp = xr.load_dataset("air_temp_lax_wl1_5_june.nc")

In [5]:
all_vars = xr.load_dataset("all_vars_lax_wl1_5_june.nc")

In [6]:
air_temp

<xarray.Dataset> Size: 6MB
Dimensions:                (simulation: 4, time: 262792)
Coordinates:
    lat                    float32 4B 33.93
    lon                    float32 4B -118.4
  * simulation             (simulation) <U26 416B 'WRF_EC-Earth3_r1i1p1f1' .....
  * time                   (time) datetime64[ns] 2MB 2000-01-01 ... 2029-12-3...
Data variables:
    Air Temperature at 2m  (simulation, time) float32 4MB 11.07 11.4 ... 18.81

In [7]:
test_data = np.arange(0, 365 * 3 * 24, 1)
test_data = np.expand_dims(test_data, [1, 2])
coords = {
    "x": 7.819e05,
    "y": -4.116e06,
    "lakemask": 0,
    "landmask": 0,
    "Lambert_Conformal": 0,
    "time": pd.date_range(start="2001-01-01-00", end="2003-12-31-23", freq="1h"),
    "scenario": ["Historical + SSP 3-7.0"],
    "simulation": ["WRF_EC-Earth3_r1i1p1f1"],
}
da = xr.DataArray(
    name="Air Temperature at 2m",
    dims=["time", "scenario", "simulation"],
    data=test_data,
    coords=coords,
)
da.attrs = {
    "variable_id": "t2",
    "extended_description": "Temperature of the air 2m above Earth's surface.",
    "units": "degC",
    "data_type": "Gridded",
    "resolution": "9 km",
    "frequency": "hourly",
    "location_subset": ["coordinate selection"],
    "approach": "Time",
    "downscaling_method": "Dynamical",
    "institution": "UCLA",
    "grid_mapping": "Lambert_Conformal",
    "timezone": "America/Los_Angeles",
}

In [8]:
da

<xarray.DataArray 'Air Temperature at 2m' (time: 26280, scenario: 1,
                                           simulation: 1)> Size: 210kB
array([[[    0]],

       [[    1]],

       [[    2]],

       ...,

       [[26277]],

       [[26278]],

       [[26279]]])
Coordinates:
    x                  float64 8B 7.819e+05
    y                  float64 8B -4.116e+06
    lakemask           int64 8B 0
    landmask           int64 8B 0
    Lambert_Conformal  int64 8B 0
  * time               (time) datetime64[ns] 210kB 2001-01-01 ... 2003-12-31T...
  * scenario           (scenario) <U22 88B 'Historical + SSP 3-7.0'
  * simulation         (simulation) <U22 88B 'WRF_EC-Earth3_r1i1p1f1'
Attributes:
    variable_id:           t2
    extended_description:  Temperature of the air 2m above Earth's surface.
    units:                 degC
    data_type:             Gridded
    resolution:            9 km
    frequency:             hourly
    location_subset:       ['coordinate selection']
    approach:              Time
    downscaling_method:    Dynamical
    institution:           UCLA
    grid_mapping:          Lambert_Conformal
    timezone:              America/Los_Angeles

In [9]:
q = 0.9
result = persistence_get_top_hours(da, q, skip_last=True) 

      📊 Processing 26,280 hours (3 years) of data
      🎯 Computing 90th percentile for each hour of year


In [28]:
test_data = np.arange(0, 365 * 3 * 24, 1)
test_data = np.expand_dims(test_data, [1, 2])
coords = {
    "x": 7.819e05,
    "y": -4.116e06,
    "lakemask": 0,
    "landmask": 0,
    "Lambert_Conformal": 0,
    "time": pd.date_range(start="2001-01-01-00", end="2003-12-31-23", freq="1h"),
    "scenario": ["Historical + SSP 3-7.0"],
    "simulation": ["sim1"],
}
test_da = xr.DataArray(
    name="Air Temperature at 2m",
    dims=["time", "scenario", "simulation"],
    data=test_data,
    coords=coords,
)
test_da.attrs = {
    "variable_id": "t2",
    "extended_description": "Temperature of the air 2m above Earth's surface.",
    "units": "degC",
    "data_type": "Gridded",
    "resolution": "9 km",
    "frequency": "hourly",
    "location_subset": ["coordinate selection"],
    "approach": "Time",
    "downscaling_method": "Dynamical",
    "institution": "UCLA",
    "grid_mapping": "Lambert_Conformal",
    "timezone": "America/Los_Angeles",
}

In [12]:
q = 0.9
# 2003 selected for all months when skip_last is False
result = persistence_get_top_hours(test_da, q, skip_last=False)

      📊 Processing 26,280 hours (3 years) of data
      🎯 Computing 90th percentile for each hour of year


In [13]:
result

,hour,sim,year
0,1,WRF_EC-Earth3_r1i1p1f1,2003
1,2,WRF_EC-Earth3_r1i1p1f1,2003
2,3,WRF_EC-Earth3_r1i1p1f1,2003
3,4,WRF_EC-Earth3_r1i1p1f1,2003
4,5,WRF_EC-Earth3_r1i1p1f1,2003
...,...,...,...
8755,8756,WRF_EC-Earth3_r1i1p1f1,2003
8756,8757,WRF_EC-Earth3_r1i1p1f1,2003
8757,8758,WRF_EC-Earth3_r1i1p1f1,2003
8758,8759,WRF_EC-Earth3_r1i1p1f1,2003


In [15]:
# 2002 selected for all months when skip_last is True
result_skip = persistence_get_top_hours(test_da, q, skip_last=True)
result_skip

      📊 Processing 26,280 hours (3 years) of data
      🎯 Computing 90th percentile for each hour of year


,hour,sim,year
0,1,WRF_EC-Earth3_r1i1p1f1,2003
1,2,WRF_EC-Earth3_r1i1p1f1,2003
2,3,WRF_EC-Earth3_r1i1p1f1,2003
3,4,WRF_EC-Earth3_r1i1p1f1,2003
4,5,WRF_EC-Earth3_r1i1p1f1,2003
...,...,...,...
8755,8756,WRF_EC-Earth3_r1i1p1f1,2002
8756,8757,WRF_EC-Earth3_r1i1p1f1,2002
8757,8758,WRF_EC-Earth3_r1i1p1f1,2002
8758,8759,WRF_EC-Earth3_r1i1p1f1,2002


In [23]:
assert (result.loc[result["hour"].between(8017, 8761)]["year"] == 2003).all()

In [22]:
assert (result_skip.loc[result["hour"].between(8017, 8761)]["year"] == 2002).all()

In [24]:
for col in ["hour", "sim", "year"]:
    assert col in result.columns
assert (np.unique(result["sim"]) == np.array(["sim1", "sim2"])).all()

AssertionError: 

In [29]:
result = persistence_get_top_hours(test_da, q, skip_last=False)
# Correctly formatted dataframe
for col in ["hour", "sim", "year"]:
    assert col in result.columns
assert (np.unique(result["sim"]) == np.array(["sim1"])).all()

      📊 Processing 26,280 hours (3 years) of data
      🎯 Computing 90th percentile for each hour of year
